# Dataset Insights (Resolution 1024)
This notebook summarizes a generated dataset of elevation-map paths and computes additional insights:
- path crossing heatmap
- elevation distributions
- path length distribution
- normalized relative elevation profile
- normalized Hausdorff distances
- identical path count


## 1. Setup and Configuration
Import libraries, configure solver, and load calibration and dataset summary files.


In [16]:
import json
import sys
import os
import math
import multiprocessing as mp
from pathlib import Path
from collections import Counter
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm
from loguru import logger as loguru_logger

# Define ROOT path first (notebook is in notebooks/)
ROOT = Path("..").resolve()

try:
    from passage.generate import _resolve_solver_config
except Exception:
    # If import fails (e.g., running in notebook without package installed),
    # temporarily prepend local `src` to sys.path and retry.
    sys.path.insert(0, str(ROOT / "src"))
    from passage.generate import _resolve_solver_config

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO");

In [18]:
# Parameters (papermill)
RESOLUTION = 1024
SPLITS = ["train", "calibration", "validation", "test"]
MAX_WORKERS = 8
DATASET_DIRPATH = "../outputs/dataset"
CONFIG_FILEPATH = "../config/config.yaml"
CALIBRATE_FILEPATH = "../outputs/calibrate/calibrate.json"

In [19]:
# Paths
CONFIG_FILEPATH = Path(CONFIG_FILEPATH)
CALIBRATE_FILEPATH = Path(CALIBRATE_FILEPATH)
DATASET_DIRPATH = Path(DATASET_DIRPATH)
RESOLUTION_DIRPATH = DATASET_DIRPATH / str(RESOLUTION)

# Solver config
with open(CONFIG_FILEPATH) as f:
    _config = yaml.safe_load(f) or {}
_solver_cfg = _config.get("solver", {})
SOLVER_CONFIG = _resolve_solver_config(_solver_cfg)

# Calibration
with open(CALIBRATE_FILEPATH) as f:
    calibration = json.load(f)

global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")

Global elevation range: [-430.0, 8767.0]


## 2. Dataset Generation
For each dataset sample, we load the stored paths and per-point elevations from metadata (no recomputation):
- **Straight-line path** between markers (baseline, generated analytically)
- **Stored no-obstacle path** (`paths.without_obstacles`) and **stored obstacle-aware path** (`paths.with_obstacles`)

Metadata now includes per-point elevations and the list of tiles used to build the crop. Metrics are computed for each sample using multiprocessing to speed up processing across CPU cores.

### Metric computations and explanations

- **Path Occurrence Heatmap**: for every sample, we increment a per-pixel counter for each pixel that belongs to the **with_obstacles** stored path. The resulting image shows frequent routing corridors.

- **Covered Tiles Visualization**: we count how often each source tile appears in `tiles_used` and visualize coverage in a lat/lon scatter plot, colored by usage frequency.

- **Start Point Elevation Distribution**: histogram of the elevation at the first pixel of each stored path. This helps understand where paths begin (e.g., high vs low terrain).

- **All Path Points Elevation Distribution**: histogram aggregating elevation values for every path pixel across all samples.

- **Path Length Distribution**: histogram of path lengths measured as number of pixels in the stored path.

- **Normalized Relative Path Elevation Profile**: for each stored path we subtract the start elevation (relative elevation) and resample the sequence to 100 uniformly spaced points in the normalized path domain [0, 1]. We then compute the mean and standard deviation across samples to show a typical elevation trend along a path.

- **Normalized Hausdorff Distance**: we compute the Hausdorff distance $H(P,Q)$ between two paths and normalize by the straight-line (marker) distance $D$ between start and end: normalized = $H(P,Q) / D$. We compute two variants:
  - **Clean vs Straight**: $P$ = path_without_obstacles, $Q$ = straight-line markers path. Measures terrain-induced deviation from straight.
  - **Obstacles vs Clean**: $P$ = path_with_obstacles, $Q$ = path_without_obstacles. Measures deviation specifically introduced by obstacles.

- **Identical Path Count**: number and percentage of samples where the stored obstacle-aware path exactly equals the stored no-obstacle path (exact pixel list equality).

In [20]:
splits = {}

start_elevations = []
all_elevations = []
path_lengths = []
normalized_profiles = []
hausdorff_clean_straight = []
hausdorff_obs_clean = []
area_ratios = []
cost_diff_percents = []

# Timing buckets
graph_prebuild_seconds = []
obstacle_generation_seconds = []
cost_compute_seconds = []
tile_loading_seconds = []
marker_search_seconds = []
marker_find_path_without_seconds = []
marker_find_path_with_seconds = []
tensor_save_seconds = []
metadata_write_seconds = []
image_conversion_seconds = []
image_write_seconds = []

# Counters
samples_processed = 0
identical_paths_count = 0

TIMING_KEYS = [
    "graph_prebuild_seconds",
    "obstacle_generation_seconds",
    "cost_compute_seconds",
    "tile_loading_seconds",
    "marker_search_seconds",
    "marker_find_path_without_seconds",
    "marker_find_path_with_seconds",
    "tensor_save_seconds",
    "metadata_write_seconds",
    "image_conversion_seconds",
    "image_write_seconds",
]

num_workers = MAX_WORKERS or os.cpu_count() or 1
try:
    num_workers = int(num_workers)
    if num_workers <= 0:
        num_workers = 1
except Exception:
    num_workers = 1
print(f"Using {num_workers} worker(s) for metadata processing")
ctx = mp.get_context("fork")

# Re-define helpers in this cell so they are picklable by process workers


def describe_stats(values, label):
    if not values:
        print(f"{label}: no data")
        return
    series = pd.Series(values, dtype="float64")
    desc = series.describe(percentiles=[0.25, 0.5, 0.75])
    std = desc.get("std", float("nan"))
    std_val = 0.0 if pd.isna(std) else std
    print(
        f"{label}: count={int(desc['count'])}, mean={desc['mean']:.2f}, std={std_val:.2f}, "
        f"min={desc['min']:.2f}, p25={desc['25%']:.2f}, median={desc['50%']:.2f}, "
        f"p75={desc['75%']:.2f}, max={desc['max']:.2f}"
    )


def _as_coords(pixels):
    coords = []
    for p in pixels or []:
        try:
            coords.append((int(p.get("i")), int(p.get("j"))))
        except Exception:
            continue
    return coords


def resample_profile(elevations, target_len=100):
    if not elevations:
        return None
    rel = np.array(elevations, dtype=float) - float(elevations[0])
    if len(rel) == 1:
        return rel.tolist() * target_len
    x_old = np.linspace(0.0, 1.0, num=len(rel))
    x_new = np.linspace(0.0, 1.0, num=target_len)
    return np.interp(x_new, x_old, rel).tolist()


def make_straight_path(start_point, end_point, n_points):
    if not start_point or not end_point or n_points <= 1:
        return []
    start = np.array([start_point.get("i", 0), start_point.get("j", 0)], dtype=float)
    end = np.array([end_point.get("i", 0), end_point.get("j", 0)], dtype=float)
    t = np.linspace(0.0, 1.0, num=n_points)
    pts = start[None, :] + (end - start)[None, :] * t[:, None]
    return [(int(round(p[0])), int(round(p[1]))) for p in pts]


def hausdorff_distance(path_a, path_b):
    if not path_a or not path_b:
        return None
    a = np.array(path_a, dtype=float)
    b = np.array(path_b, dtype=float)
    dists = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(axis=2))
    return float(max(dists.min(axis=1).max(), dists.min(axis=0).max()))


def relative_area_between_paths(path_a, path_b, marker_dist, samples=100):
    if marker_dist is None or marker_dist <= 0 or not path_a or not path_b:
        return None

    def _resample(path):
        pts = np.array(path, dtype=float)
        if len(pts) == 1:
            return np.repeat(pts, samples, axis=0)
        x_old = np.linspace(0.0, 1.0, num=len(pts))
        x_new = np.linspace(0.0, 1.0, num=samples)
        i_vals = np.interp(x_new, x_old, pts[:, 0])
        j_vals = np.interp(x_new, x_old, pts[:, 1])
        return np.stack([i_vals, j_vals], axis=1)

    a_rs, b_rs = _resample(path_a), _resample(path_b)
    dists = np.linalg.norm(a_rs - b_rs, axis=1)
    area = np.trapezoid(dists, x=np.linspace(0.0, 1.0, num=samples))
    return float(area / (marker_dist**2))


def _process_metadata(md_path: Path):
    try:
        with open(md_path) as f:
            md = yaml.safe_load(f) or {}
    except Exception:
        return None

    tile_counts = Counter()
    for tile in md.get("tile", {}).get("overlapping", []):
        try:
            lat = int(tile.get("lat"))
            lon = int(tile.get("lon"))
            tile_counts[(lat, lon)] += 1
        except Exception:
            continue

    clean_pixels = md.get("paths", {}).get("without_obstacles", {}).get("pixels", []) or []
    obs_pixels = md.get("paths", {}).get("with_obstacles", {}).get("pixels", []) or []
    clean_coords = _as_coords(clean_pixels)
    obs_coords = _as_coords(obs_pixels)

    grid_free_counts = Counter(clean_coords)
    grid_obs_counts = Counter(obs_coords)

    identical = 1 if clean_coords and obs_coords and clean_coords == obs_coords else 0

    obs_elevs = [p.get("elevation") for p in obs_pixels if p.get("elevation") is not None]
    start_vals = []
    all_vals = []
    profile = None
    if obs_elevs:
        start_vals.append(float(obs_elevs[0]))
        all_vals.extend(float(v) for v in obs_elevs)
        profile = resample_profile([float(v) for v in obs_elevs])

    path_length = len(obs_coords)

    marker_start = md.get("markers", {}).get("start_point", {}) or {}
    marker_end = md.get("markers", {}).get("end_point", {}) or {}
    try:
        marker_dist = math.dist(
            (float(marker_start.get("i")), float(marker_start.get("j"))),
            (float(marker_end.get("i")), float(marker_end.get("j"))),
        )
    except Exception:
        marker_dist = None

    h_clean_norm = None
    h_obs_norm = None
    area_val = None
    if marker_dist and marker_dist > 0:
        straight_path = make_straight_path(marker_start, marker_end, max(len(clean_coords), 2))
        h_clean = hausdorff_distance(clean_coords, straight_path)
        if h_clean is not None:
            h_clean_norm = float(h_clean / marker_dist)
        h_obs = hausdorff_distance(obs_coords, clean_coords)
        if h_obs is not None:
            h_obs_norm = float(h_obs / marker_dist)
        area_val = relative_area_between_paths(obs_coords, clean_coords, marker_dist)

    cost_percent = md.get("paths", {}).get("with_obstacles", {}).get("cost_diff_percent")
    cost_percent = float(cost_percent) if cost_percent is not None else None

    timing_md = md.get("timing", {}) or {}
    timing_vals = {k: timing_md.get(k) for k in TIMING_KEYS}

    return {
        "tile_overlapping": tile_counts,
        "grid_free_counts": grid_free_counts,
        "grid_obs_counts": grid_obs_counts,
        "identical_paths": identical,
        "samples_processed": 1,
        "start_elevations": start_vals,
        "all_elevations": all_vals,
        "profile": profile,
        "path_length": path_length,
        "hausdorff_clean": h_clean_norm,
        "hausdorff_obs": h_obs_norm,
        "area_ratio": area_val,
        "cost_percent": cost_percent,
        "timing": timing_vals,
    }


for split in SPLITS:
    # Dataset paths and summary (derived from parameters)
    split_dirpath = RESOLUTION_DIRPATH / split
    metadata_dirpath = split_dirpath / "metadata"
    tensors_dirpath = split_dirpath / "tensors"
    images_dirpath = split_dirpath / "images"

    metadata_filepaths = sorted(metadata_dirpath.rglob("**/*.yaml"))

    tile_overlapping = Counter()
    grid_free = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int32)
    grid_obstacles = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int32)

    if metadata_filepaths:
        effective_workers = min(num_workers, len(metadata_filepaths))
        with ProcessPoolExecutor(max_workers=effective_workers, mp_context=ctx) as executor:
            for res in tqdm(
                executor.map(_process_metadata, metadata_filepaths),
                total=len(metadata_filepaths),
                desc=f"Processing {split} metadata (parallel)",
            ):
                if not res:
                    continue

                tile_overlapping.update(res["tile_overlapping"])

                for (i, j), cnt in res["grid_free_counts"].items():
                    grid_free[i, j] += cnt
                for (i, j), cnt in res["grid_obs_counts"].items():
                    grid_obstacles[i, j] += cnt

                identical_paths_count += res["identical_paths"]
                samples_processed += res["samples_processed"]

                start_elevations.extend(res["start_elevations"])
                all_elevations.extend(res["all_elevations"])

                if res["profile"] is not None:
                    normalized_profiles.append(res["profile"])

                path_lengths.append(res["path_length"])

                if res["hausdorff_clean"] is not None:
                    hausdorff_clean_straight.append(res["hausdorff_clean"])
                if res["hausdorff_obs"] is not None:
                    hausdorff_obs_clean.append(res["hausdorff_obs"])
                if res["area_ratio"] is not None:
                    area_ratios.append(res["area_ratio"])
                if res["cost_percent"] is not None:
                    cost_diff_percents.append(res["cost_percent"])

                for key, val in res["timing"].items():
                    if val is None:
                        continue
                    if key == "graph_prebuild_seconds":
                        graph_prebuild_seconds.append(val)
                    elif key == "obstacle_generation_seconds":
                        obstacle_generation_seconds.append(val)
                    elif key == "cost_compute_seconds":
                        cost_compute_seconds.append(val)
                    elif key == "tile_loading_seconds":
                        tile_loading_seconds.append(val)
                    elif key == "marker_search_seconds":
                        marker_search_seconds.append(val)
                    elif key == "marker_find_path_without_seconds":
                        marker_find_path_without_seconds.append(val)
                    elif key == "marker_find_path_with_seconds":
                        marker_find_path_with_seconds.append(val)
                    elif key == "tensor_save_seconds":
                        tensor_save_seconds.append(val)
                    elif key == "metadata_write_seconds":
                        metadata_write_seconds.append(val)
                    elif key == "image_conversion_seconds":
                        image_conversion_seconds.append(val)
                    elif key == "image_write_seconds":
                        image_write_seconds.append(val)

    # save summary info
    splits[split] = {
        "name": split,
        "num_samples": len(metadata_filepaths),
        "tiles_overlapping": {k: {"lat": k[0], "lon": k[1], "count": v} for k, v in tile_overlapping.items()},
        "grids": {
            "free": grid_free,
            "obstacles": grid_obstacles,
        },
    }

Using 8 worker(s) for metadata processing


Processing test metadata (parallel): 100%|██████████| 10/10 [00:02<00:00,  3.39it/s]


## 3. Dataset Summary
Report basic statistics: requested vs. processed samples, and success rate.


In [21]:
print(f"Resolution: {RESOLUTION}")

for split, data in splits.items():
    print(f"\nSplit: {split}")
    print(f"Number of samples: {data['num_samples']}")
    print("Number of unique overlapping tiles:", len(data["tiles_overlapping"]))

Resolution: 1024

Split: train
Number of samples: 70
Number of unique overlapping tiles: 139

Split: calibration
Number of samples: 10
Number of unique overlapping tiles: 23

Split: validation
Number of samples: 10
Number of unique overlapping tiles: 21

Split: test
Number of samples: 10
Number of unique overlapping tiles: 22


## 4. Covered Tiles Visualization
Each point represents a source tile used to build the elevation crops. The color indicates how many samples used the tile.

In [22]:
# Covered Tiles Visualization: build a 180x360 (1° per pixel) grid image from metadata overlapping tiles
# Zero-based full-globe grid: rows=180 (lat), cols=360 (lon); lat range [-90,89], lon range [-180,179]
for split, summary in splits.items():
    grid = np.zeros((180, 360), dtype=np.int32) * np.nan

    for (lat, lont), cell_info in summary["tiles_overlapping"].items():
        lat_i = int(lat)
        lon_j = int(lont)
        cnt = int(cell_info.get("count", 0))
        i = 89 - lat_i  # row 0 = northmost lat (89)
        j = lon_j + 180
        grid[i, j] = cnt

    fig = plt.figure(figsize=(14, 7))

    cmap = cm.viridis.copy()
    cmap.set_bad(color="white")  # zeros shown as white

    plt.imshow(grid, origin="upper", extent=[-180, 179, -90, 89], cmap=cmap, aspect="equal")
    plt.title(f"Covered Tiles for Split: {split} (180x360 grid, 1° per pixel)")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.colorbar(label="Tile Overlap Count")
    plt.show()

## 4. Path Occurrence Heatmaps
Two heatmaps aggregate pixel crossing counts across the dataset:
- Left: Clean (no obstacles) stored path crossings
- Right: With Obstacles stored path crossings


In [23]:
for split, summary in splits.items():
    for k, grid in summary["grids"].items():
        label = "without obstacles" if k == "free" else "with obstacles"
        plt.figure(figsize=(14, 14))
        plt.title(f"Aggregated paths for split {split} ({label})")
        plt.imshow(grid, cmap="hot", origin="upper")
        plt.axis("off")
        plt.tight_layout()

## 5. Start Point Elevation Distribution
Histogram of the absolute elevation at the first point of each obstacle-aware path.

In [24]:
vals = [float(v) for v in start_elevations if v is not None]
if vals:
    describe_stats(vals, "Start point elevations (m)")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=30, edgecolor="black")
    plt.xlabel("Elevation (m)")
    plt.ylabel("Count")
    plt.title("Start Point Elevation Distribution")
    plt.tight_layout()
    plt.show()
else:
    print("No start elevations found; ensure metadata files are available.")

Start point elevations (m): count=100, mean=4044.04, std=2779.51, min=-327.00, p25=1634.00, median=3489.50, p75=6664.75, max=8738.00


## 6. All Path Points Elevation Distribution
Histogram of all elevations visited by obstacle-aware paths.

In [25]:
vals = [v for v in all_elevations if v is not None]
if vals:
    describe_stats(vals, "All Path Point Elevations")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=50, edgecolor="black")
    plt.xlabel("Elevation (m)")
    plt.ylabel("Count")
    plt.title("All Path Point Elevations")
    plt.tight_layout()
    plt.show()
else:
    print("No path elevation data available.")

All Path Point Elevations: count=63509, mean=3295.90, std=2058.47, min=-327.00, p25=1591.00, median=3354.00, p75=4857.00, max=8738.00


## 7. Path Length Distribution
Histogram of obstacle-aware path lengths (number of pixels).

In [26]:
vals = [v for v in path_lengths if v is not None]
if vals:
    describe_stats(vals, "Path Lengths (pixels)")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=30, edgecolor="black")
    plt.xlabel("Length (pixels)")
    plt.ylabel("Count")
    plt.title("Path Lengths")
    plt.tight_layout()
    plt.show()
else:
    print("No path length data available.")

Path Lengths (pixels): count=100, mean=635.09, std=293.57, min=59.00, p25=433.50, median=582.00, p75=813.50, max=1479.00


## 8. Normalized Relative Path Elevation Profile
Each path elevation sequence is shifted by its start elevation and resampled to a common length $[0,1]$.
We then plot the mean and standard deviation across samples.

In [27]:
if normalized_profiles:
    profiles_arr = np.array(normalized_profiles, dtype=np.float64)
    mean_profile = np.mean(profiles_arr, axis=0)
    std_profile = np.std(profiles_arr, axis=0)
    x_axis = np.linspace(0, 1, profiles_arr.shape[1])

    plt.figure(figsize=(10, 6))
    plt.plot(x_axis, mean_profile, "b-", linewidth=2, label="Mean Relative Elevation")
    plt.fill_between(
        x_axis, mean_profile - std_profile, mean_profile + std_profile, alpha=0.2, color="blue", label="Std Dev"
    )
    plt.xlabel("Normalized Path Progress")
    plt.ylabel("Relative Elevation (m)")
    plt.title("Normalized Relative Elevation Profile")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No normalized profiles available.")

## 9. Hausdorff Distance: Path vs. Straight Line
Normalized Hausdorff distance between the optimal (no-obstacle) path and the straight-line path between markers. This measures how much terrain elevation causes deviations from a straight line.

In [28]:
clean_vals = [v for v in hausdorff_clean_straight if v is not None and v > 0]
if clean_vals:
    describe_stats(clean_vals, "Normalized Hausdorff: Clean vs Straight")
    plt.figure(figsize=(8, 5))
    plt.hist(clean_vals, bins=30, edgecolor="black")
    plt.xlabel("Normalized Distance")
    plt.ylabel("Count")
    plt.title("Normalized Hausdorff: Clean vs Straight Line (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive Hausdorff data for clean vs straight.")

Normalized Hausdorff: Clean vs Straight: count=100, mean=0.21, std=0.18, min=0.01, p25=0.11, median=0.17, p75=0.25, max=1.35


## 10. Hausdorff Distance: Obstacles vs. No Obstacles
Normalized Hausdorff distance between the obstacle-aware path and the no-obstacle path on the same elevation map.

In [29]:
obs_vals = [v for v in hausdorff_obs_clean if v is not None and v > 0]
if obs_vals:
    describe_stats(obs_vals, "Normalized Hausdorff: Obstacles vs Clean")
    plt.figure(figsize=(8, 5))
    plt.hist(obs_vals, bins=100, edgecolor="black")
    plt.xlabel("Normalized Distance")
    plt.ylabel("Count")
    plt.title("Normalized Hausdorff: Obstacles vs No Obstacles (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive Hausdorff data for obstacles vs clean.")

Normalized Hausdorff: Obstacles vs Clean: count=44, mean=0.20, std=0.17, min=0.01, p25=0.07, median=0.17, p75=0.31, max=0.80


## 11. Area Ratio & Cost Difference
Area ratio = (area between paths in pixels) / (marker distance in pixels)^2. Cost difference = 100 * (cost_with - cost_without) / cost_without.

In [30]:
# Area ratio distribution and cost difference visualization
area_pos = [v for v in area_ratios if v is not None and v > 0]
if area_pos:
    describe_stats(area_pos, "Area Ratio (positive)")
    plt.figure(figsize=(8, 5))
    plt.hist(area_pos, bins=40, edgecolor="black")
    plt.xlabel("Area Ratio")
    plt.ylabel("Count")
    plt.title("Area Ratio: (area between paths) / (marker_dist^2) (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive area ratio data available.")

# Cost difference percentage distribution
cd_vals = [v for v in cost_diff_percents if v is not None and v > 0]
if cd_vals:
    describe_stats(cd_vals, "Cost Difference Percent (positive)")
    plt.figure(figsize=(8, 5))
    plt.hist(cd_vals, bins=50, edgecolor="black")
    plt.xlabel("Percent")
    plt.ylabel("Count")
    plt.title("Cost Difference Percent: 100*(cost_with - cost_without)/cost_without")
    plt.tight_layout()
    plt.show()
else:
    print("No cost difference percentage data available.")

Area Ratio (positive): count=44, mean=0.00, std=0.00, min=0.00, p25=0.00, median=0.00, p75=0.00, max=0.00


Cost Difference Percent (positive): count=44, mean=31.60, std=42.13, min=0.19, p25=3.75, median=16.30, p75=40.52, max=196.54


## 11b. Timing Summary

Plot per-sample timing distributions to identify hotspots:
- `tile_loading_seconds`
- `marker_search_seconds`
- `marker_find_path_without_seconds`
- `marker_find_path_with_seconds`

- `graph_prebuild_seconds`

- `obstacle_generation_seconds`- `image_write_seconds`

- `cost_compute_seconds`- `image_conversion_seconds`

- `tensor_save_seconds`- `metadata_write_seconds`

In [31]:
# Timing distributions and stats
gp_vals = [v for v in graph_prebuild_seconds if v is not None and v >= 0]
og_vals = [v for v in obstacle_generation_seconds if v is not None and v >= 0]
cc_vals = [v for v in cost_compute_seconds if v is not None and v >= 0]
tl_vals = [v for v in tile_loading_seconds if v is not None and v >= 0]
ms_vals = [v for v in marker_search_seconds if v is not None and v >= 0]
mpw_vals = [v for v in marker_find_path_without_seconds if v is not None and v >= 0]
mpt_vals = [v for v in marker_find_path_with_seconds if v is not None and v >= 0]
ts_vals = [v for v in tensor_save_seconds if v is not None and v >= 0]
mw_vals = [v for v in metadata_write_seconds if v is not None and v >= 0]
ic_vals = [v for v in image_conversion_seconds if v is not None and v >= 0]
iw_vals = [v for v in image_write_seconds if v is not None and v >= 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if gp_vals:
    axes[0].hist(gp_vals, bins=30, edgecolor="black")
    axes[0].set_title("Graph prebuild (s)")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Count")
else:
    axes[0].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[0].set_title("Graph prebuild (s)")

if og_vals:
    axes[1].hist(og_vals, bins=30, edgecolor="black")
    axes[1].set_title("Obstacle generation (s)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Count")
else:
    axes[1].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[1].set_title("Obstacle generation (s)")

if cc_vals:
    axes[2].hist(cc_vals, bins=30, edgecolor="black")
    axes[2].set_title("Cost compute (s)")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Count")
else:
    axes[2].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[2].set_title("Cost compute (s)")

fig.suptitle("Per-sample Timing Distributions")
plt.tight_layout()
plt.show()

# Additional timing plots for new metrics (if present)
new_metrics = [
    ("Tile load", tl_vals),
    ("Marker search", ms_vals),
    ("Marker find (no obstacles)", mpw_vals),
    ("Marker find (with obstacles)", mpt_vals),
    ("Tensor save", ts_vals),
    ("Metadata write", mw_vals),
    ("Image conversion", ic_vals),
    ("Image write", iw_vals),
]
new_metrics = [(name, vals) for name, vals in new_metrics if vals]
if new_metrics:
    n2 = len(new_metrics)
    cols2 = 3
    rows2 = math.ceil(n2 / cols2)
    fig2, axes2 = plt.subplots(rows2, cols2, figsize=(5 * cols2, 4 * rows2))
    ax_list2 = axes2.flatten() if hasattr(axes2, "flatten") else [axes2]
    for ax in ax_list2[n2:]:
        ax.axis("off")
    for ax, (name, vals) in zip(ax_list2, new_metrics):
        ax.hist(vals, bins=30, edgecolor="black")
        ax.set_title(f"{name} (n={len(vals)})")
        ax.set_xlabel("seconds")
    fig2.suptitle("Additional Timing Distributions")
    plt.tight_layout()
    plt.show()

    # Print summary stats for the timing buckets (if any exist)
    if not (gp_vals or og_vals or cc_vals):
        print("No timing data available.")
    else:
        if gp_vals:
            describe_stats(gp_vals, "Graph prebuild seconds")
        if og_vals:
            describe_stats(og_vals, "Obstacle generation seconds")
        if cc_vals:
            describe_stats(cc_vals, "Cost compute seconds")

Graph prebuild seconds: count=100, mean=0.00, std=0.00, min=0.00, p25=0.00, median=0.00, p75=0.00, max=0.00
Obstacle generation seconds: count=100, mean=1.78, std=1.38, min=0.10, p25=0.79, median=1.45, p75=2.34, max=5.81
Cost compute seconds: count=100, mean=0.04, std=0.02, min=0.01, p25=0.01, median=0.03, p75=0.06, max=0.09


## 12. Identical Path Count
Number of cases where the obstacle-aware path is exactly the same as the no-obstacle path.

In [32]:
if samples_processed > 0:
    pct_identical = (identical_paths_count / samples_processed) * 100
else:
    pct_identical = 0.0

print(f"Identical paths: {identical_paths_count} / {samples_processed} ({pct_identical:.1f}%)")

labels = ["Identical", "Different"]
counts = [identical_paths_count, max(samples_processed - identical_paths_count, 0)]
colors = ["#4caf50", "#f44336"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, counts, color=colors, edgecolor="black")
axes[0].set_ylabel("Samples")
axes[0].set_title("Path Equality Counts")

if samples_processed > 0:
    axes[1].pie(counts, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Identical vs Different")
    axes[1].axis("equal")
else:
    axes[1].axis("off")
    axes[1].text(0.5, 0.5, "No samples", ha="center", va="center")

plt.tight_layout()
plt.show()

Identical paths: 56 / 100 (56.0%)
